In [1]:
# If needed
# !pip install openai pandas tqdm

import json
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
with open("assessments_catalog.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

df.head()

,name,url,pdf_text,job_level,test_language,adaptive_support,description,duration,test_type,remote_support
0,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,.NET Framework 4.5 Assessment Fact Sheet Overv...,"Professional Individual Contributor, Mid-Profe...",English (USA),No,The.NET Framework 4.5 test measures knowledge ...,30.0,[Knowledge & Skills],Yes
1,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,This report is designed to be given to individ...,"Mid-Professional, Professional Individual Cont...",English (USA),No,This report is designed to be given to individ...,NaN,"[Ability & Aptitude, Personality & Behavior, B...",Yes
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,.NET MVC Assessment Fact Sheet Overview Multi-...,"Mid-Professional, Professional Individual Cont...",English (USA),No,Multi-choice test that measures the knowledge ...,17.0,[Knowledge & Skills],Yes
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,.NET MVVM. Assessment Fact Sheet. Overview. Mu...,"Mid-Professional, Professional Individual Cont...",English (USA),No,Multi-choice test that measures the knowledge ...,5.0,[Knowledge & Skills],Yes
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,.NET WCF. Assessment Fact Sheet. Overview. Mul...,"Mid-Professional, Professional Individual Cont...",English (USA),No,Multi-choice test that measures the knowledge ...,11.0,[Knowledge & Skills],Yes


In [3]:
def clean_job_levels(level_str):
    if pd.isna(level_str):
        return []
    return [lvl.strip() for lvl in level_str.split(",")]

df["job_level_list"] = df["job_level"].apply(clean_job_levels)

df[["job_level", "job_level_list"]].head()

,job_level,job_level_list
0,"Professional Individual Contributor, Mid-Profe...","[Professional Individual Contributor, Mid-Prof..."
1,"Mid-Professional, Professional Individual Cont...","[Mid-Professional, Professional Individual Con..."
2,"Mid-Professional, Professional Individual Cont...","[Mid-Professional, Professional Individual Con..."
3,"Mid-Professional, Professional Individual Cont...","[Mid-Professional, Professional Individual Con..."
4,"Mid-Professional, Professional Individual Cont...","[Mid-Professional, Professional Individual Con..."


In [4]:
JOB_LEVEL_MAPPING = {
    "Graduate": (0, 1),
    "Entry-Level": (0, 2),
    "General Population": (0, 3),
    
    "Supervisor": (2, 5),
    "Front Line Manager": (3, 6),
    "Professional Individual Contributor": (3, 6),
    "Mid-Professional": (4, 7),
    "Manager": (5, 8),
    
    "Executive": (8, 12),
    "Director": (10, 15),
}

In [5]:
def compute_experience_numeric(level_list):
    ranges = []
    
    for level in level_list:
        if level in JOB_LEVEL_MAPPING:
            ranges.append(JOB_LEVEL_MAPPING[level])
    
    if not ranges:
        return None
    
    # Merge ranges
    min_years = min(r[0] for r in ranges)
    max_years = max(r[1] for r in ranges)
    
    midpoint = (min_years + max_years) / 2
    
    return {
        "min_years": min_years,
        "max_years": max_years,
        "midpoint": midpoint
    }

df["experience_numeric"] = df["job_level_list"].apply(compute_experience_numeric)

df[["job_level_list", "experience_numeric"]].head()

,job_level_list,experience_numeric
0,"[Professional Individual Contributor, Mid-Prof...","{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}"
1,"[Mid-Professional, Professional Individual Con...","{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}"
2,"[Mid-Professional, Professional Individual Con...","{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}"
3,"[Mid-Professional, Professional Individual Con...","{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}"
4,"[Mid-Professional, Professional Individual Con...","{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}"


In [6]:
df["duration"] = pd.to_numeric(df["duration"], errors="coerce")

df["duration"].describe()

count    285.000000
mean      12.849123
std        8.534625
min        0.000000
25%        8.000000
50%       10.000000
75%       15.000000
max       60.000000
Name: duration, dtype: float64

In [7]:
def create_base_text(row):
    parts = []
    
    if pd.notna(row["description"]):
        parts.append(f"Description: {row['description']}")
        
    if pd.notna(row["pdf_text"]):
        parts.append(f"PDF Info: {row['pdf_text']}")
        
    if row["test_type"]:
        parts.append(f"Test Type: {', '.join(row['test_type'])}")
        
    return "\n".join(parts)

df["raw_combined_text"] = df.apply(create_base_text, axis=1)

df["raw_combined_text"].iloc[0][:500]

'Description: The.NET Framework 4.5 test measures knowledge of .NET environment. Designed for experienced users, this test covers the following topics: Application Development, Application Foundation, Data Modeling, Deployment, Diagnostics, Performance, Portability, and Security.\nPDF Info: .NET Framework 4.5 Assessment Fact Sheet Overview The .NET Framework 4.5 test measures knowledge of .NET environment. Designed for experienced users. Job Family/Title Programmers, Application Developers. Detail'

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="")

In [9]:
def extract_structured_info(text):
    
    prompt = f"""
    Extract structured information from this assessment description.

    Return JSON with keys:
    - assessed_skills (list)
    - target_roles (list)
    - cognitive_dimensions (list)
    - industry_tags (list)
    - extracted_duration_minutes (number or null)

    TEXT:
    {text}
    """
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a structured information extraction assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.choices[0].message.content

In [12]:
sample_text = df["raw_combined_text"].iloc[0]

structured_output = extract_structured_info(sample_text)

print(structured_output)

```json
{
  "assessed_skills": [
    "Application Development",
    "Application Foundation",
    "Data Modeling",
    "Deployment",
    "Diagnostics",
    "Performance",
    "Portability",
    "Security"
  ],
  "target_roles": [
    "Programmers",
    "Application Developers"
  ],
  "cognitive_dimensions": [],
  "industry_tags": [
    "Information Technology"
  ],
  "extracted_duration_minutes": 26
}
```


In [16]:
import json
import re

def safe_json_parse(text):
    try:
        return json.loads(text)
    except:
        # Try extracting JSON block
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                return None
        return None

In [17]:
def extract_structured_info(text):
    
    prompt = f"""
    Extract structured information from this assessment content.

    Return ONLY valid JSON with this exact structure:

    {{
        "assessed_skills": [],
        "target_roles": [],
        "cognitive_dimensions": [],
        "industry_tags": [],
        "extracted_duration_minutes": null
    }}

    TEXT:
    {text}
    """
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "You extract structured assessment metadata."},
            {"role": "user", "content": prompt}
        ]
    )
    
    content = response.choices[0].message.content
    return safe_json_parse(content)

In [18]:
structured_results = []

for text in tqdm(df["raw_combined_text"]):
    result = extract_structured_info(text)
    structured_results.append(result)

df["structured_info"] = structured_results

100%|██████████| 377/377 [19:17<00:00,  3.07s/it]


In [19]:
df["assessed_skills"] = df["structured_info"].apply(
    lambda x: x.get("assessed_skills") if x else []
)

df["target_roles"] = df["structured_info"].apply(
    lambda x: x.get("target_roles") if x else []
)

df["cognitive_dimensions"] = df["structured_info"].apply(
    lambda x: x.get("cognitive_dimensions") if x else []
)

df["industry_tags"] = df["structured_info"].apply(
    lambda x: x.get("industry_tags") if x else []
)

df["extracted_duration"] = df["structured_info"].apply(
    lambda x: x.get("extracted_duration_minutes") if x else None
)

In [20]:
df["final_duration"] = df["duration"]

df.loc[df["final_duration"].isna(), "final_duration"] = df["extracted_duration"]

df["final_duration"] = pd.to_numeric(df["final_duration"], errors="coerce")

In [21]:
def normalize_list(items):
    if not items:
        return []
    return list(set([i.strip().lower() for i in items if isinstance(i, str)]))

df["assessed_skills_norm"] = df["assessed_skills"].apply(normalize_list)
df["cognitive_dimensions_norm"] = df["cognitive_dimensions"].apply(normalize_list)
df["target_roles_norm"] = df["target_roles"].apply(normalize_list)

In [22]:
def create_embedding_text(row):
    
    parts = []
    
    if row["assessed_skills_norm"]:
        parts.append("Skills: " + ", ".join(row["assessed_skills_norm"]))
    
    if row["cognitive_dimensions_norm"]:
        parts.append("Cognitive: " + ", ".join(row["cognitive_dimensions_norm"]))
    
    if row["target_roles_norm"]:
        parts.append("Target Roles: " + ", ".join(row["target_roles_norm"]))
    
    if row["industry_tags"]:
        parts.append("Industry: " + ", ".join(row["industry_tags"]))
    
    if pd.notna(row["description"]):
        parts.append("Summary: " + row["description"])
    
    return "\n".join(parts)

df["embedding_text"] = df.apply(create_embedding_text, axis=1)

df["embedding_text"].iloc[0]

'Skills: portability, application development, performance, diagnostics, security, application foundation, deployment, data modeling\nTarget Roles: programmers, application developers\nIndustry: Information Technology\nSummary: The.NET Framework 4.5 test measures knowledge of .NET environment. Designed for experienced users, this test covers the following topics: Application Development, Application Foundation, Data Modeling, Deployment, Diagnostics, Performance, Portability, and Security.'

In [23]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

In [24]:
embeddings = []

for text in tqdm(df["embedding_text"]):
    emb = get_embedding(text)
    embeddings.append(emb)

df["embedding"] = embeddings

100%|██████████| 377/377 [01:42<00:00,  3.67it/s]


In [26]:
final_columns = [
    "name",
    "url",
    "assessed_skills_norm",
    "cognitive_dimensions_norm",
    "target_roles_norm",
    "industry_tags",
    "final_duration",
    "experience_numeric",
    "embedding"
]

df_final = df[final_columns].copy()

In [27]:
def clean_for_json(row):
    
    if isinstance(row["embedding"], np.ndarray):
        row["embedding"] = row["embedding"].tolist()
        
    if isinstance(row["final_duration"], np.generic):
        row["final_duration"] = float(row["final_duration"])
    
    return row

df_final = df_final.apply(clean_for_json, axis=1)

In [28]:
df_final.to_json(
    "processed_assessments.json",
    orient="records",
    indent=2,
    force_ascii=False
)

In [1]:
import json

with open("processed_assessments.json", "r", encoding="utf-8") as f:
    data = json.load(f)

len(data)

377

In [2]:
for item in data:
    if "url" in item and item["url"]:
        item["url"] = item["url"].replace("\\/", "/")

In [4]:
with open("processed_assessments_clean.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [5]:
sample_query = """ICICI Bank Assistant Admin, Experience required 0-2 years, test should be 30-40 mins long
"""

results = recommend_assessments(sample_query, top_n=10)

display(
    results[[
        "name",
        "url",
        "assessed_skills_norm",
        "target_roles_norm",
        "final_duration",
        "final_score"
    ]]
)

NameError: name 'recommend_assessments' is not defined